In [1]:
import torch
import json
import os
import requests
import numpy as np
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification, AutoModel
from huggingface_hub import hf_hub_download
import torch.nn as nn

# --- CONFIGURATION ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NULL_TOKEN = "[NULL]"
MAX_LEN = 128

# REPLACE THESE WITH YOUR HUGGING FACE REPO IDs
EXTRACTOR_REPO = "affan002/xlm-roberta-extractor-eng-zho-wout-aug"
PAIRING_REPO   = "affan002/roberta-pairing-task2-eng"
REGRESSOR_REPO = "affan002/asp-opi-regressor" 

# --- MODEL CLASS DEFINITION (Required for loading Regressor) ---
class TransformerVARegressor(nn.Module):
    def __init__(self, model_name, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.reg_head = nn.Linear(self.backbone.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0]
        x = self.dropout(cls_output)
        return self.reg_head(x)

# --- DATASET TASKS ---
TASKS = [
    {"lang": "eng", "domain": "laptop", "url": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/subtask_2/eng/eng_laptop_dev_task2.jsonl"},
    {"lang": "eng", "domain": "restaurant", "url": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/subtask_2/eng/eng_restaurant_dev_task2.jsonl"},
    {"lang": "zho", "domain": "laptop", "url": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/subtask_2/zho/zho_laptop_dev_task2.jsonl"},
    {"lang": "zho", "domain": "restaurant", "url": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/subtask_2/zho/zho_restaurant_dev_task2.jsonl"}
    
    ]

In [2]:
from huggingface_hub import hf_hub_download , login
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import json
import torch

# Login with your token for private models
login(token=os.environ["HF_TOKEN"])  # <--- Add your token here
print("✅ Logged in to Hugging Face")

# --- 1. Load Extractor ---
print("Loading Extractor...")
ext_tokenizer = AutoTokenizer.from_pretrained(EXTRACTOR_REPO)
ext_model = AutoModelForTokenClassification.from_pretrained(EXTRACTOR_REPO).to(DEVICE)
ext_model.eval()
id2label = ext_model.config.id2label

# --- 2. Load Pairer ---
print("Loading Pairer...")
pair_tokenizer = AutoTokenizer.from_pretrained(PAIRING_REPO)
pair_model = AutoModelForSequenceClassification.from_pretrained(PAIRING_REPO).to(DEVICE)
pair_model.eval()

# --- 3. Load Regressor (Custom Class) ---
print("Loading Regressor...")
# We initialize with base XLM-R structure first
reg_model = TransformerVARegressor(model_name="xlm-roberta-large") # Or 'base' if you used base
reg_tokenizer = AutoTokenizer.from_pretrained(REGRESSOR_REPO)

# Resize embeddings (Crucial!)
reg_model.backbone.resize_token_embeddings(len(reg_tokenizer))

# Download weights manually because it's a custom class
try:
    # Try safetensors first
    from safetensors.torch import load_file
    model_path = hf_hub_download(repo_id=REGRESSOR_REPO, filename="model.safetensors")
    state_dict = load_file(model_path)
except:
    # Fallback to pytorch_model.bin
    model_path = hf_hub_download(repo_id=REGRESSOR_REPO, filename="pytorch_model.bin")
    state_dict = torch.load(model_path, map_location=DEVICE)

reg_model.load_state_dict(state_dict)
reg_model.to(DEVICE)
reg_model.eval()

print("✅ All Models Loaded Successfully!")

✅ Logged in to Hugging Face
Loading Extractor...


tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


pytorch_model.bin:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

✅ All Models Loaded Successfully!


## Function for extractor model

In [3]:
def run_extractor(text):
    # 1. Prepend Double NULLs (Crucial: Match Training!)
    aug_text = f"{NULL_TOKEN} {NULL_TOKEN} {text}"
    
    inputs = ext_tokenizer(
        aug_text, 
        return_tensors="pt", 
        truncation=True, 
        max_length=128
    ).to(DEVICE)
    
    with torch.no_grad():
        outputs = ext_model(**inputs)
    
    # Get Predictions
    preds = torch.argmax(outputs.logits, dim=2)[0].cpu().numpy()
    tokens = ext_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    
    # Helper to clean RoBERTa tokens
    def clean(toks): return ext_tokenizer.convert_tokens_to_string(toks).strip()

    aspects, opinions = [], []
    curr_word, curr_type = [], None
    
    for t, p in zip(tokens, preds):
        label = id2label[p]
        
        # Skip special tokens (CLS, SEP, PAD)
        if t in [ext_tokenizer.cls_token, ext_tokenizer.sep_token, ext_tokenizer.pad_token]:
            continue
        
        # --- EXPLICIT NULL CHECK ---
        # If the model tagged the special [NULL] token, we record it immediately.
        # This handles Implicit Aspects/Opinions.
        if t == NULL_TOKEN:
            if "ASP" in label: aspects.append("[NULL]")
            if "OPI" in label: opinions.append("[NULL]")
            continue # Don't add [NULL] to current_word list
        
        # Standard BIO Logic
        if label.startswith("B-"):
            if curr_word: 
                w = clean(curr_word)
                if curr_type == "ASP": aspects.append(w)
                elif curr_type == "OPI": opinions.append(w)
            curr_word = [t]
            curr_type = label.split("-")[1] # 'ASP' or 'OPI'
            
        elif label.startswith("I-") and curr_type == label.split("-")[1]:
            curr_word.append(t)
            
        else: # 'O' tag or mismatch
            if curr_word:
                w = clean(curr_word)
                if curr_type == "ASP": aspects.append(w)
                elif curr_type == "OPI": opinions.append(w)
            curr_word = []; curr_type = None
            
    # Catch the last word
    if curr_word:
        w = clean(curr_word)
        if curr_type == "ASP": aspects.append(w)
        elif curr_type == "OPI": opinions.append(w)
        
    return list(set(aspects)), list(set(opinions))

In [4]:
# --- TEST SUITE for run_extractor
test_cases = [
    {
        "desc": "Standard Case (Explicit)",
        "text": "The battery life is amazing.",
        "expected_asp": "battery life",
        "expected_opi": "amazing"
    },
    {
        "desc": "Implicit Aspect (Should detect [NULL])",
        "text": "Too expensive for what you get.",
        "expected_asp": "[NULL]",
        "expected_opi": "Too expensive"
    },
    {
        "desc": "Implicit Opinion (Should detect [NULL])",
        "text": "The screen resolution is 1080p.",
        "expected_asp": "screen resolution",
        "expected_opi": "[NULL]" 
    },
    {
        "desc": "Multiple Triplets",
        "text": "The food was great but the service was slow.",
        "expected_asp": "food, service",
        "expected_opi": "great, slow"
    }
]



for case in test_cases:
    text = case["text"]
    print(f"--- Test: {case['desc']} ---")
    print(f"Input: \"{text}\"")
    
    # Run your function
    pred_aspects, pred_opinions = run_extractor(text)
    
    print(f"Predicted Aspects: {pred_aspects}")
    print(f"Predicted Opinions: {pred_opinions}")
    
    # Visual Check
    asp_check = "✅" if case["expected_asp"] in str(pred_aspects) else "⚠️"
    print(f"   Status: {asp_check}\n")

--- Test: Standard Case (Explicit) ---
Input: "The battery life is amazing."
Predicted Aspects: ['battery life']
Predicted Opinions: ['amazing']
   Status: ✅

--- Test: Implicit Aspect (Should detect [NULL]) ---
Input: "Too expensive for what you get."
Predicted Aspects: ['[NULL]']
Predicted Opinions: ['Too expensive']
   Status: ✅

--- Test: Implicit Opinion (Should detect [NULL]) ---
Input: "The screen resolution is 1080p."
Predicted Aspects: ['screen']
Predicted Opinions: ['[NULL]']
   Status: ⚠️

--- Test: Multiple Triplets ---
Input: "The food was great but the service was slow."
Predicted Aspects: ['service', 'food']
Predicted Opinions: ['slow', 'great']
   Status: ⚠️



## Function for Pairing model

In [5]:



def run_pairer(text, aspect, opinion):
    
    # Input A: [NULL] [NULL] Text
    aug_text = f"{NULL_TOKEN} {NULL_TOKEN} {text}"
    
    # Input B: Aspect </s> Opinion
    pair_text = f"{aspect} {pair_tokenizer.sep_token} {opinion}"
    
    inputs = pair_tokenizer(
        aug_text, 
        pair_text, 
        return_tensors="pt", 
        truncation=True, 
        max_length=MAX_LEN
    ).to(DEVICE)
    
    with torch.no_grad():
        outputs = pair_model(**inputs)
    
    # Label 1 = Valid Pair
    probs = torch.softmax(outputs.logits, dim=1)[0]
    pred_label = torch.argmax(probs).item()
    
    # Debug Info (Optional: helps you see confidence)
    print(f"   Pair: {aspect} + {opinion} -> Score: {probs[1]:.4f}")

    return pred_label == 1

In [6]:
print("Testing Pairing Model Logic...\n")

# Define a tricky sentence with two distinct pairs
test_sentence = "The screen is bright but the battery life is short."

test_cases = [
    # Case A: Valid Pairs (Should be True)
    {"asp": "screen", "opi": "bright", "expected": True},
    {"asp": "battery life", "opi": "short", "expected": True},
    
    # Case B: Hard Negatives (Swapped - Should be False)
    {"asp": "screen", "opi": "short", "expected": False},
    {"asp": "battery life", "opi": "bright", "expected": False},
    
    # Case C: Random/Unrelated (Should be False)
    {"asp": "screen", "opi": "delicious", "expected": False},
]

for case in test_cases:
    result = run_pairer(test_sentence, case["asp"], case["opi"])
    
    status = "✅ PASS" if result == case["expected"] else "❌ FAIL"
    label_str = "Valid" if result else "Invalid"
    
    print(f"Pair: ({case['asp']}, {case['opi']})")
    print(f"   Predicted: {label_str} | {status}\n")

Testing Pairing Model Logic...

   Pair: screen + bright -> Score: 0.9997
Pair: (screen, bright)
   Predicted: Valid | ✅ PASS

   Pair: battery life + short -> Score: 0.9997
Pair: (battery life, short)
   Predicted: Valid | ✅ PASS

   Pair: screen + short -> Score: 0.0014
Pair: (screen, short)
   Predicted: Invalid | ✅ PASS

   Pair: battery life + bright -> Score: 0.0009
Pair: (battery life, bright)
   Predicted: Invalid | ✅ PASS

   Pair: screen + delicious -> Score: 0.0003
Pair: (screen, delicious)
   Predicted: Invalid | ✅ PASS



## Regressor model function

In [7]:
def run_regressor(text, asp_txt, opi_txt):
    # 1. Handle NULLs
    
    # 2. Prepare Input A: Sentence (with NULL tokens to match Extractor context)
    # (Matches the VADataset logic we defined)
    text_a = f"{NULL_TOKEN} {NULL_TOKEN} {text}"
    
    # 3. Prepare Input B: Aspect + Separator + Opinion
    # We manually insert the separator token here so it becomes "Aspect [SEP] Opinion"
    text_b = f"{asp_txt} {reg_tokenizer.sep_token} {opi_txt}"
    
    # 4. Tokenize using pairs
    inputs = reg_tokenizer(
        text=text_a,
        text_pair=text_b,  # Pass the Aspect/Opinion pair here
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(DEVICE)
    
    with torch.no_grad():
        outputs = reg_model(inputs["input_ids"], inputs["attention_mask"])
        
    scores = outputs.cpu().numpy()[0]
    v = max(1.0, min(9.0, scores[0]))
    a = max(1.0, min(9.0, scores[1]))
    
    return f"{v:.2f}#{a:.2f}"

In [8]:
# --- Test Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("🧪 Testing Regressor Model (Triplet Input)...\n")

# --- Test Case 1: Standard Input ---
# "The battery life is absolutely amazing."
text_1 = "The battery life is absolutely amazing."
aspect_1 = "battery life"
opinion_1 = "amazing"
# Note: Domain is no longer needed for this model

score_1 = run_regressor(text_1, aspect_1, opinion_1)
print(f"Test 1 (Explicit):")
print(f"   Input:   Sentence='{text_1}' | Pair='{aspect_1} + {opinion_1}'")
print(f"   Output:  {score_1}\n")

# --- Test Case 2: Negative Input ---
# "The service was slow and rude."
text_2 = "The service was slow and rude."
aspect_2 = "service"
opinion_2 = "slow"

score_2 = run_regressor(text_2, aspect_2, opinion_2)
print(f"Test 2 (Negative):")
print(f"   Input:   Sentence='{text_2}' | Pair='{aspect_2} + {opinion_2}'")
print(f"   Output:  {score_2}\n")

# --- Test Case 3: Implicit Aspect ---
# "Too expensive for what you get."
text_3 = "Too expensive for what you get."
aspect_3 = "[NULL]" 
opinion_3 = "Too expensive"

score_3 = run_regressor(text_3, aspect_3, opinion_3)
print(f"Test 3 (Implicit):")
print(f"   Input:   Sentence='{text_3}' | Pair='{aspect_3} + {opinion_3}'")
print(f"   Output:  {score_3}\n")

# --- Verification Checks ---
if "#" in score_1 and len(score_1.split("#")) == 2:
    print("Output format check passed (V#A)")
else:
    print("Output format Invalid!")

🧪 Testing Regressor Model (Triplet Input)...

Test 1 (Explicit):
   Input:   Sentence='The battery life is absolutely amazing.' | Pair='battery life + amazing'
   Output:  7.92#7.92

Test 2 (Negative):
   Input:   Sentence='The service was slow and rude.' | Pair='service + slow'
   Output:  2.76#7.13

Test 3 (Implicit):
   Input:   Sentence='Too expensive for what you get.' | Pair='[NULL] + Too expensive'
   Output:  3.51#6.20

Output format check passed (V#A)


In [9]:
def run_test_case(sentence, pairs_to_check):
    """
    Tests multiple Aspect-Opinion pairs against a single sentence.
    
    Args:
        sentence (str): The review text.
        pairs_to_check (list): List of tuples (Aspect, Opinion, Expected_Boolean)
                               e.g. [("food", "good", True), ("food", "bad", False)]
    """
    print(f"\n📝 Sentence: \"{sentence}\"")
    
    # 1. Prepare Input A (Sentence with Double NULL)
    # Must match training format!
    augmented_text = f"{NULL_TOKEN} {NULL_TOKEN} {sentence}"
    
    for asp, opi, expected in pairs_to_check:
        # 2. Prepare Input B (Aspect + Separator + Opinion)
        # We pass the raw strings (even "[NULL]") because that's what we trained on
        pair_text = f"{asp} {pair_tokenizer.sep_token} {opi}"
        
        # 3. Tokenize
        inputs = pair_tokenizer(
            augmented_text, 
            pair_text, 
            truncation=True, 
            padding="max_length", 
            max_length=128,
            return_tensors="pt"
        ).to(DEVICE)
        
        # 4. Inference
        with torch.no_grad():
            outputs = pair_model(**inputs)
            
        probs = torch.softmax(outputs.logits, dim=1)[0]
        valid_prob = probs[1].item() # Probability of Class 1 (Valid)
        is_valid = torch.argmax(probs).item() == 1
        
        # 5. Formatting Output
        status_icon = "✅ VALID" if is_valid else "❌ INVALID"
        match_icon = "🎯" if is_valid == expected else "⚠️ WRONG"
        
        print(f"   ({asp}, {opi}) -> {status_icon}  (Conf: {valid_prob:.1%}) | Expected: {expected} {match_icon}")

## Pipeline

In [10]:
def fetch_data(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        return [json.loads(line) for line in response.text.strip().split('\n') if line]
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return []

In [11]:
TASKS = [
    {"lang": "eng", "domain": "laptop", "url": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/subtask_2/eng/eng_laptop_dev_task2.jsonl"},
    {"lang": "eng", "domain": "restaurant", "url": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/subtask_2/eng/eng_restaurant_dev_task2.jsonl"},
    {"lang": "zho", "domain": "laptop", "url": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/subtask_2/zho/zho_laptop_dev_task2.jsonl"},
    {"lang": "zho", "domain": "restaurant", "url": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/subtask_2/zho/zho_restaurant_dev_task2.jsonl"}
    
    ]

output_dir = "subtask_2"
os.makedirs(output_dir, exist_ok=True)

print(f"Starting Inference Pipeline...")

for task in TASKS:
    lang = task['lang']
    domain = task['domain']
    filename = f"pred_{lang}_{domain}.jsonl"
    print(f"\nProcessing {lang}-{domain}...")
    
    # 1. Load Data from GitHub
    data = fetch_data(task['url'])
    results = []
    
    for entry in tqdm(data):
        text = entry['Text']
        
        # --- STEP 1: EXTRACT ---
        # Get lists of candidates
        aspects, opinions = run_extractor(text)
        
        triplets = []
        
        # --- STEP 2: PAIR LOOP (Cartesian Product) ---
        for asp in aspects:
            for opi in opinions:
                
                # # Skip if both are NULL (usually noise)
                # if asp == "[NULL]" and opi == "[NULL]": continue
                
                # --- STEP 3: VALIDATE ---
                # Ask Model 2: "Is this pair valid?"
                if run_pairer(text, asp, opi):
                    
                    # --- STEP 4: SCORE ---
                    # Ask Model 3: "What is the VA?"
                    va_score = run_regressor(text, asp, opi)
                    
                    # Clean format for output
                    final_asp = "NULL" if asp == "[NULL]" else asp
                    final_opi = "NULL" if opi == "[NULL]" else opi
                    
                    triplets.append({
                        "Aspect": final_asp,
                        "Opinion": final_opi,
                        "VA": va_score
                    })
        
        results.append({
            "ID": entry['ID'],
            "Triplet": triplets
        })
        
    # Save to File
    with open(f"{output_dir}/{filename}", 'w', encoding='utf-8') as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("\n----------Inference Complete!---------")

# --- ZIP FOR SUBMISSION ---
import shutil
shutil.make_archive("subtask_2", 'zip', output_dir)
print(f"Ready: submission_task2.zip")

Starting Inference Pipeline...

Processing eng-laptop...


  0%|          | 0/200 [00:00<?, ?it/s]

   Pair: perforemce + Great -> Score: 0.9997
   Pair: color gamut + wide -> Score: 0.9997
   Pair: color gamut + Very bright -> Score: 0.0012
   Pair: display + wide -> Score: 0.0041
   Pair: display + Very bright -> Score: 0.9996
   Pair: Battery life + bad -> Score: 0.9997
   Pair: Chromebook + very clean -> Score: 0.9997
   Pair: laptop's screen + very bright -> Score: 0.9996
   Pair: laptop's screen + clear -> Score: 0.9988
   Pair: laptop's screen + brighter than -> Score: 0.9965
   Pair: sound quality + like -> Score: 0.0294
   Pair: sound quality + excellent -> Score: 0.9994
   Pair: video card + very much worth -> Score: 0.9996
   Pair: laptop + Great -> Score: 0.9996
   Pair: Zenbook + [NULL] -> Score: 0.0006
   Pair: Zenbook + powerful -> Score: 0.9997
   Pair: memory + [NULL] -> Score: 0.7944
   Pair: memory + powerful -> Score: 0.9715
   Pair: storage + [NULL] -> Score: 0.5712
   Pair: storage + powerful -> Score: 0.9906
   Pair: RAM + borderline -> Score: 0.9997
   Pair: l

  0%|          | 0/200 [00:00<?, ?it/s]

   Pair: coffee + great -> Score: 0.9996
   Pair: Food + great -> Score: 0.9996
   Pair: Customer service + awesome -> Score: 0.0017
   Pair: Customer service + fantastic -> Score: 0.9997
   Pair: food + awesome -> Score: 0.9994
   Pair: food + fantastic -> Score: 0.0013
   Pair: Shrimp taco's + perfectly -> Score: 0.9991
   Pair: Rolls + PACKED with -> Score: 0.9193
   Pair: Rolls + artfully made -> Score: 0.9996
   Pair: Rolls + reasonably priced -> Score: 0.9996
   Pair: service + slow -> Score: 0.9996
   Pair: restaurant + very clean -> Score: 0.9991
   Pair: restaurant + good -> Score: 0.0040
   Pair: restaurant + tasty -> Score: 0.0026
   Pair: restaurant + pretty attentive -> Score: 0.0015
   Pair: service + very clean -> Score: 0.0027
   Pair: service + good -> Score: 0.0016
   Pair: service + tasty -> Score: 0.0014
   Pair: service + pretty attentive -> Score: 0.9996
   Pair: food + very clean -> Score: 0.0049
   Pair: food + good -> Score: 0.9978
   Pair: food + tasty -> Scor

  0%|          | 0/300 [00:00<?, ?it/s]

   Pair: cpu + 比i7高一截 -> Score: 0.9039
   Pair: cpu + 根本沒比12700h強多少 -> Score: 0.8418
   Pair: 價格 + 比i7高一截 -> Score: 0.6584
   Pair: 價格 + 根本沒比12700h強多少 -> Score: 0.6249
   Pair: 12900h + 比i7高一截 -> Score: 0.5184
   Pair: 12900h + 根本沒比12700h強多少 -> Score: 0.8052
   Pair: 風扇噪音 + 值得讚賞 -> Score: 0.0173
   Pair: 風扇噪音 + 瑕不掩 -> Score: 0.0052
   Pair: 風扇噪音 + 算是 -> Score: 0.5316
   Pair: 風扇噪音 + 比較大 -> Score: 0.0035
   Pair: 風扇噪音 + 相當強悍 -> Score: 0.5939
   Pair: 風扇噪音 + 很輕 -> Score: 0.6376
   Pair: 硬體規格 + 值得讚賞 -> Score: 0.3236
   Pair: 硬體規格 + 瑕不掩 -> Score: 0.1943
   Pair: 硬體規格 + 算是 -> Score: 0.7858
   Pair: 硬體規格 + 比較大 -> Score: 0.1987
   Pair: 硬體規格 + 相當強悍 -> Score: 0.8822
   Pair: 硬體規格 + 很輕 -> Score: 0.7828
   Pair: 散熱 + 值得讚賞 -> Score: 0.1116
   Pair: 散熱 + 瑕不掩 -> Score: 0.0603
   Pair: 散熱 + 算是 -> Score: 0.4751
   Pair: 散熱 + 比較大 -> Score: 0.0593
   Pair: 散熱 + 相當強悍 -> Score: 0.4970
   Pair: 散熱 + 很輕 -> Score: 0.5851
   Pair: RazerBlade162024 + 值得讚賞 -> Score: 0.1118
   Pair: RazerBlade162024 + 瑕不掩 -> Sc

  0%|          | 0/300 [00:00<?, ?it/s]

   Pair: 豬肉 + 舒肥 -> Score: 0.9413
   Pair: 豬肉 + 不錯吃 -> Score: 0.9953
   Pair: 雞肉 + 舒肥 -> Score: 0.8838
   Pair: 雞肉 + 不錯吃 -> Score: 0.9848
   Pair: 叉燒 + 舒肥 -> Score: 0.6600
   Pair: 叉燒 + 不錯吃 -> Score: 0.9900
   Pair: 扁豆 + 舒肥 -> Score: 0.8307
   Pair: 扁豆 + 不錯吃 -> Score: 0.9960
   Pair: 玉米 + 舒肥 -> Score: 0.8767
   Pair: 玉米 + 不錯吃 -> Score: 0.9969
   Pair: 食物 + 很好吃 -> Score: 0.9991
   Pair: 食物 + 很不錯 -> Score: 0.9950
   Pair: 外送 + 很好吃 -> Score: 0.9682
   Pair: 外送 + 很不錯 -> Score: 0.9962
   Pair:  + 很不錯 -> Score: 0.9492
   Pair:  + 很Creamy -> Score: 0.9977
   Pair: 調味 + 很不錯 -> Score: 0.9837
   Pair: 調味 + 很Creamy -> Score: 0.9925
   Pair:  + 無味 -> Score: 0.6930
   Pair:  + 鹹甜適中 -> Score: 0.9839
   Pair: 油膏 + 無味 -> Score: 0.3329
   Pair: 油膏 + 鹹甜適中 -> Score: 0.9865
   Pair: 滷味 + 無味 -> Score: 0.0904
   Pair: 滷味 + 鹹甜適中 -> Score: 0.8294
   Pair: 魚卵炒飯 +  -> Score: 0.6053
   Pair: 魚卵炒飯 + 好吃 -> Score: 0.9063
   Pair: 魚卵炒飯 + 非常推薦 -> Score: 0.9925
   Pair: 魚卵炒飯 + 滿滿的 -> Score: 0.9047
   Pair: 魚卵炒飯 + 超讚 -

In [12]:
## testing some manually 
text = "everyone is super friendly and the flavors are good"
aspects, opinons = run_extractor("everyone is super friendly and the flavors are good")

for a in aspects:
    for o in opinions:
        print(run_pairer(text, a,o))

   Pair: [NULL] + 甘美綿密 -> Score: 0.0005
False
   Pair: flavors + 甘美綿密 -> Score: 0.0076
False
